# Supervisor Pattern


Supervisor Pattern [Step 06 - Central Routing to Specialist Agents]

> **MLCourse - Agentic AI - LangGraph**

The supervisor pattern uses a central coordinator (the "supervisor") that
inspects the user request and routes it to the appropriate specialist agent.
Each specialist handles one domain (research, writing, etc.) and returns
results back to the supervisor, who decides whether to continue or finish.

# What you will learn

1. How to implement a supervisor node that uses tools to route to specialists.
2. Building specialist agent nodes with `create_agent`.
3. Tool-based handoff: the supervisor calls a tool whose name matches a node.
4. Visualizing the full supervisor graph.

### Key takeaways

- The supervisor is just another node; it uses an LLM to pick the next agent.
- Specialist agents are independent subgraphs (tools, prompts, LLM calls).
- The supervisor loop continues until the supervisor decides to finish.

### Setup: imports, environment


In [ ]:
import os                           # env access
from dotenv import load_dotenv      # .env loading

load_dotenv(override=False)         # load without overriding

from typing import Annotated, TypedDict, Literal  # typed state
from langgraph.graph import StateGraph, END        # graph primitives
from langchain.agents import create_agent          # agent factory (LangChain v1+)
from langchain_ollama import ChatOllama            # local LLM
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage  # messages
from langchain_core.tools import tool              # tool decorator

# NOTE: `create_agent` used to live in `langgraph.prebuilt`. As of
# LangGraph v1.0 it moved and was renamed: use `create_agent` from
# `langchain.agents`. The `prompt=` argument is now `system_prompt=`.


### Model guard: check Ollama


In [ ]:
try:                                        # quick connectivity test
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("Ollama reachable -- full supervisor pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("Ollama not reachable:", exc)
    print("Graph structure demonstrated without LLM calls")


### Define specialist tools (research and writing)


In [ ]:
@tool
def research(query: str) -> str:
    """Research a topic and return a summary of findings.

    Args:
        query: The research question or topic to investigate.
    """
    # In production this would call a search API or knowledge base.
    return ("Research findings for '%s': "
            "Key facts include historical context, current trends, "
            "and expert opinions." % query)

@tool
def write(content: str) -> str:
    """Write or draft content based on provided information.

    Args:
        content: The information or outline to turn into written content.
    """
    # In production this would generate formatted text using an LLM.
    return ("Written draft: "
            "This is a well-structured piece covering %s" % content)


### Define the supervisor state with a step counter


In [ ]:
class SupervisorState(TypedDict):
    """State carries the conversation, routing decision, and step counter."""
    messages: Annotated[list, "conversation messages"]
    next: str                              # routing decision: agent name or FINISH
    step: int                              # safety counter to prevent infinite loops


### Build specialist agents using create_agent


In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None

# Research agent: uses the research tool
research_agent = create_agent(
    model=llm if LLM_AVAILABLE else ChatOllama(model="llama3.1:8b", temperature=0),
    tools=[research],                      # only has the research tool
    system_prompt="You are a research specialist. Use the research tool to find information.",
) if LLM_AVAILABLE else None

# Writing agent: uses the write tool
write_agent = create_agent(
    model=llm if LLM_AVAILABLE else ChatOllama(model="llama3.1:8b", temperature=0),
    tools=[write],                         # only has the write tool
    system_prompt="You are a writing specialist. Use the write tool to draft content.",
) if LLM_AVAILABLE else None

print("Specialist agents created:",
      "research + write" if LLM_AVAILABLE else "structure only")


### Supervisor node: routes to specialists or finishes


In [ ]:
def supervisor_node(state: SupervisorState) -> dict:
    """Supervisor examines the conversation and decides the next step."""
    step = state.get("step", 0) + 1       # increment step counter

    system_prompt = (
        "You are a supervisor managing two agents:\n"
        "- research: for finding information\n"
        "- write: for drafting content\n"
        "- FINISH: when the task is complete\n\n"
        "Based on the conversation, decide which agent should act next.\n"
        "Respond with ONLY the agent name or FINISH."
    )

    if LLM_AVAILABLE:                      # use LLM for routing decisions
        messages = [SystemMessage(content=system_prompt)] + state["messages"]
        response = llm.invoke(messages)    # get routing decision
        decision = response.content.strip().lower()  # normalize

        if "research" in decision:
            next_agent = "researcher"
        elif "write" in decision:
            next_agent = "writer"
        else:
            next_agent = "FINISH"          # default: finish if unclear
    else:                                  # fallback: fixed demo sequence
        # Without an LLM we demonstrate a hardcoded two-hop sequence:
        # step 1 -> researcher, step 2 -> writer, step 3 -> FINISH
        demo_sequence = ["researcher", "writer", "FINISH"]
        idx = min(step - 1, len(demo_sequence) - 1)
        next_agent = demo_sequence[idx]

    print("[supervisor] routing to:", next_agent, "(step %d)" % step)
    return {"next": next_agent, "step": step}  # return routing + counter


### Specialist node wrappers: run the sub-agents


In [ ]:
def researcher_node(state: SupervisorState) -> dict:
    """Research specialist: gathers information on the topic."""
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="[researcher] Research complete (simulated)")]}

    print("[researcher] starting research...")
    result = research_agent.invoke(state)  # run the research agent
    last_msg = result["messages"][-1].content if result.get("messages") else "done"
    print("[researcher] done:", last_msg[:60])
    return {"messages": [AIMessage(content="[Research] %s" % last_msg)]}

def writer_node(state: SupervisorState) -> dict:
    """Writing specialist: drafts content from provided info."""
    if not LLM_AVAILABLE:
        return {"messages": [AIMessage(content="[writer] Draft complete (simulated)")]}

    print("[writer] starting writing...")
    result = write_agent.invoke(state)     # run the writing agent
    last_msg = result["messages"][-1].content if result.get("messages") else "done"
    print("[writer] done:", last_msg[:60])
    return {"messages": [AIMessage(content="[Writing] %s" % last_msg)]}


### Routing function: conditional edge from supervisor


In [ ]:
def route_supervisor(state: SupervisorState) -> str:
    """Read the 'next' field from state and return the target node name."""
    return state.get("next", "FINISH")    # default to FINISH if missing


### Build the supervisor graph


In [ ]:
builder = StateGraph(SupervisorState)      # create builder

builder.add_node("supervisor", supervisor_node)   # routing coordinator
builder.add_node("researcher", researcher_node)   # research specialist
builder.add_node("writer", writer_node)           # writing specialist

builder.set_entry_point("supervisor")     # supervisor is always first

builder.add_conditional_edges(
    "supervisor",                          # source node
    route_supervisor,                      # function that reads state
    {                                      # map: decision -> target node
        "researcher": "researcher",
        "writer": "writer",
        "FINISH": END,
    },
)

# Specialists always return to the supervisor
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

graph = builder.compile()                 # finalize

print("Supervisor graph compiled: supervisor -> {researcher, writer, FINISH}")


### Visualize the supervisor graph


In [ ]:
from IPython.display import Image, display

try:                                        # wrap in try/except for offline
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Visualization unavailable:", exc)
    print("Nodes:", list(builder.nodes.keys()))


### Run the supervisor pipeline


In [ ]:
print("=== Running supervisor pipeline ===")
print()

initial_state = {
    "messages": [
        HumanMessage(content="Research the latest trends in AI agents and write a brief summary.")
    ],
    "next": "",                            # empty initially; supervisor fills it
    "step": 0,                             # start at step 0
}

for step in graph.stream(initial_state):
    for node_name, node_output in step.items():
        if node_name == "supervisor":
            print("[supervisor] decided:", node_output.get("next", "unknown"))
        else:
            msgs = node_output.get("messages", [])
            for m in msgs:
                content = m.content if hasattr(m, "content") else str(m)
                print("[%s] %s" % (node_name, content[:80]))

print()
print("NOTEBOOK COMPLETE: supervisor pattern demonstrated successfully")
